# Support Vector Machines — Practice Set 2 🎯
## Mr Beast Videos: Engagement Classification

### Rules
- `alpha = 0.05`, `random_state = 617`
- Use `train` unless stated otherwise
- Do NOT round; no commas; lead with 0
- **Always scale features before SVM**

### About the Data
`mr_beast.csv` — YouTube video statistics.
- `views`, `likes`, `comments`, `duration_seconds`, `number_of_tags`, `length_description`, `length_title`, `time_since`

> **Goal 1 (Classification)**: Create a 3-class engagement label based on `likes`, then classify videos.
> **Goal 2 (Regression)**: Use SVR to predict `likes` from other features.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR, LinearSVC
from sklearn.metrics import accuracy_score, classification_report, root_mean_squared_error

data = pd.read_csv('mr_beast.csv')
print(data.shape)
print(data.describe()[['views','likes','comments','time_since']])

---
## ❓ Question 1
> Create a 3-class engagement variable:
> ```python
> data['engagement'] = pd.cut(data.likes,
>     bins=[0, 50000, 500000, float('inf')],
>     labels=['low', 'medium', 'high'])
> ```
> Separate outcome (`engagement`) from features: `views`, `comments`, `duration_seconds`, `number_of_tags`, `length_description`, `length_title`, `time_since`.
>
> Split 70/30 with `random_state=617`.
>
> **What is the distribution of `engagement` in the train sample?** (as counts)

In [ ]:
data['engagement'] = pd.cut(data.likes,
    bins=[0, 50000, 500000, float('inf')],
    labels=['low', 'medium', 'high'])

y = data.engagement
X = data[['views','comments','duration_seconds','number_of_tags',
          'length_description','length_title','time_since']]

X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
    X, y, train_size=0.7, random_state=617)

print('Class distribution in train:')
print(y_tr.value_counts())

---
## ❓ Question 2
> Scale the features using `StandardScaler`. Fit on train, transform both.
>
> **What is the standard deviation of `views` in the scaled train set?** (Should be ≈1.0)

In [ ]:
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr_raw)
X_te = scaler.transform(X_te_raw)

print('Std of scaled views in train:', X_tr[:, 0].std())

---
## ❓ Question 3
> Fit a **Linear SVM** classifier (`LinearSVC(C=1, random_state=617)`) on the 3-class engagement problem.
>
> **What is the train accuracy?**
> **What is the test accuracy?**

In [ ]:
svm_lin = LinearSVC(C=1, random_state=617)
svm_lin.fit(X_tr, y_tr)

print('Linear SVM train accuracy:', svm_lin.score(X_tr, y_tr))
print('Linear SVM test accuracy: ', svm_lin.score(X_te, y_te))

---
## ❓ Question 4
> Fit an **RBF SVM** classifier (`SVC(kernel='rbf', C=1, random_state=617)`).
>
> **What is the train accuracy?**
> **What is the test accuracy?**
> **Does RBF outperform Linear SVM on the test set?**

In [ ]:
svm_rbf = SVC(kernel='rbf', C=1, random_state=617)
svm_rbf.fit(X_tr, y_tr)

print('RBF SVM train accuracy:', svm_rbf.score(X_tr, y_tr))
print('RBF SVM test accuracy: ', svm_rbf.score(X_te, y_te))

---
## ❓ Question 5
> Use `GridSearchCV` with 5-fold CV to tune `C` and `gamma` for the RBF SVM.
>
> Use: `C = [0.1, 1, 10, 100]` and `gamma = [1e-3, 1e-2, 0.1, 1]`.
>
> **What is the best parameter combination?**
> **What is the best CV score?**
> **What are the train and test accuracies of the best model?**

In [ ]:
param_grid = {'C': [0.1, 1, 10, 100], 'gamma': [1e-3, 1e-2, 0.1, 1]}
grid = GridSearchCV(SVC(kernel='rbf', random_state=617), param_grid, cv=5, n_jobs=-1)
grid.fit(X_tr, y_tr)

print('Best params:   ', grid.best_params_)
print('Best CV score: ', grid.best_score_)
print('Train accuracy:', grid.score(X_tr, y_tr))
print('Test accuracy: ', grid.score(X_te, y_te))

---
## ❓ Question 6
> Print the **classification report** for the best model on the test set.
>
> **Which class has the highest precision?**
> **Which class has the lowest recall?**

In [ ]:
print(classification_report(y_te, grid.predict(X_te)))

---
## ❓ Question 7
## Part 2: SVR — Regression with `likes` as the target

> Now use `likes` as the **numeric** outcome and fit an **SVR** (Support Vector Regression).
> Use features: `views`, `comments`, `duration_seconds`, `number_of_tags`, `time_since`.
> Split 70/30 with `random_state=617`. Scale features.
>
> Fit `SVR(kernel='rbf')` with default parameters. Call it `svr_default`.
> **What is the train RMSE of `svr_default`?**

In [ ]:
# New setup for regression
y_reg = data.likes
X_reg = data[['views','comments','duration_seconds','number_of_tags','time_since']]

X_tr2_raw, X_te2_raw, y_tr2, y_te2 = train_test_split(
    X_reg, y_reg, train_size=0.7, random_state=617)

scaler2 = StandardScaler()
X_tr2 = scaler2.fit_transform(X_tr2_raw)
X_te2 = scaler2.transform(X_te2_raw)

svr_default = SVR(kernel='rbf')
svr_default.fit(X_tr2, y_tr2)

print('SVR Train RMSE:', root_mean_squared_error(y_tr2, svr_default.predict(X_tr2)))
print('SVR Test RMSE: ', root_mean_squared_error(y_te2, svr_default.predict(X_te2)))

---
## ❓ Question 8
> Tune the SVR using `GridSearchCV` with 5-fold CV.
>
> Use: `C = [0.1, 1, 10, 100]` and `gamma = [0.01, 0.1, 1]`.
>
> **What is the best `C` and `gamma`?**
> **What is the best CV R² score?** (GridSearchCV default scoring for SVR is R²)

In [ ]:
svr_grid = GridSearchCV(
    SVR(kernel='rbf'),
    {'C': [0.1, 1, 10, 100], 'gamma': [0.01, 0.1, 1]},
    cv=5, n_jobs=-1)
svr_grid.fit(X_tr2, y_tr2)

print('Best params:  ', svr_grid.best_params_)
print('Best CV R²:   ', svr_grid.best_score_)
print('Train RMSE:   ', root_mean_squared_error(y_tr2, svr_grid.predict(X_tr2)))
print('Test RMSE:    ', root_mean_squared_error(y_te2, svr_grid.predict(X_te2)))

---
## ❓ Question 9
> **What is the predicted `likes` for the first observation in the test set using the tuned SVR?**

In [ ]:
first_pred = svr_grid.predict(X_te2[[0]])
print('Predicted likes for first test obs:', first_pred[0])
print('Actual likes:                      ', y_te2.iloc[0])

---
## ❓ Question 10
> **Compare the tuned SVR against the default SVR** by test RMSE.
>
> **Did tuning improve the test RMSE?**
>
> Also conceptually: **When would you choose SVR over linear regression?**

In [ ]:
rmse_default = root_mean_squared_error(y_te2, svr_default.predict(X_te2))
rmse_tuned   = root_mean_squared_error(y_te2, svr_grid.predict(X_te2))

print(f'Default SVR test RMSE: {rmse_default:.2f}')
print(f'Tuned SVR test RMSE:   {rmse_tuned:.2f}')
print(f'Improvement: {rmse_default - rmse_tuned:.2f}')

# SVR preferred when: non-linear relationship exists,
# data is small to medium sized,
# you want robustness to outliers

---
# ✅ Key Concepts

| Topic | Detail |
|---|---|
| Multi-class SVM | SVM extends to multi-class via One-vs-All or One-vs-One |
| SVR | SVM for regression — same kernel trick, same scaling requirement |
| GridSearchCV scoring | Default is accuracy (classification) or R² (regression) |
| C parameter | Controls margin softness — too high = overfit |
| gamma parameter | Controls boundary complexity — too high = overfit |

## SVR Pipeline
```python
from sklearn.svm import SVR
from sklearn.metrics import root_mean_squared_error

svr = SVR(kernel='rbf', C=1, gamma='scale')
svr.fit(X_train, y_train)     # y_train is numeric (not binary)

pred = svr.predict(X_test)
rmse = root_mean_squared_error(y_test, pred)
```